In [8]:
import numpy as np

In [9]:
rng = np.random.default_rng(42)
m_por_classe = 50
X0 = rng.normal(loc=[-1, -1], scale=1.2, size=(m_por_classe, 2))
X1 = rng.normal(loc=[1, 1], scale=1.2, size=(m_por_classe, 2))

X_dados = np.vstack([X0, X1])
y = np.concatenate([np.zeros(m_por_classe), np.ones(m_por_classe)])

idx = rng.permutation(len(y))
corte = int(0.8 * len(y))
treino, teste = idx[:corte], idx[corte:]

uns = np.ones((X_dados.shape[0], 1))
X = np.hstack([uns, X_dados])

X_treino, y_treino = X[treino], y[treino]
X_teste, y_teste = X[teste], y[teste]
beta = np.zeros(X.shape[1])

In [10]:
class RegressaoLogistica():
    def __init__(self, alpha=0.1, epocas=1000, tol=1e-8):
        # Taxa de aprendizado e épocas
        self.alpha = alpha
        self.epocas = epocas
        self.tol = tol
        self.beta_ = None
        self.historico_ = []

    def adicionar_intercepto(self, X):
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        if not np.allclose(X[:, 0], 1.0): 
            return np.column_stack([np.ones(len(X)), X])
        return X

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        # Treina e guarda o histórico de custo
        Xb = self.adicionar_intercepto(X)
        m = len(y)
        self.beta_ = np.zeros(Xb.shape[1])
        self.historico_ = []

        for t in range(self.epocas):
            z = Xb @ self.beta_
            p = self.sigmoid(z)
            # Custo (Entropia Cruzada)
            p_clip = np.clip(p, 1e-12, 1 - 1e-12)
            custo = -np.mean(y * np.log(p_clip) + (1 - y) * np.log(1 - p_clip))
            self.historico_.append(custo)
            gradiente = (Xb.T @ (p - y)) / m
            self.beta_ = self.beta_ - self.alpha * gradiente
            if t > 0 and abs(self.historico_[-2] - self.historico_[-1]) < self.tol:
                break
        return self
    def predict_proba(self, X):
        # Probabilidades da classe 1
        Xb = self.adicionar_intercepto(X)
        return self.sigmoid(Xb @ self.beta_)
    def predict(self, X, limiar=0.5):
        # Classes 0/1 com limiar 0.5
        return (self.predict_proba(X) >= limiar).astype(int)
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

In [12]:
# Instanciando e treinando o modelo
modelo = RegressaoLogistica(alpha=0.1, epocas=1000)
modelo.fit(X_treino, y_treino)

print("Épocas executadas:", len(modelo.historico_))
print(f"Custo inicial: {modelo.historico_[0]:.4f} -> Custo final: {modelo.historico_[-1]:.4f}")
print(f"Acurácia no Treino: {modelo.score(X_treino, y_treino):.3f}")
print(f"Acurácia no Teste: {modelo.score(X_teste, y_teste):.3f}")

Épocas executadas: 1000
Custo inicial: 0.6931 -> Custo final: 0.1862
Acurácia no Treino: 0.900
Acurácia no Teste: 0.900
